# Módulo 00 — o modelo do curso

Um XGBoost e uma regressão logística sobre o Ouro COVID
([`GOLD.md`](../GOLD.md), [`gold/MANIFEST.md`](../gold/MANIFEST.md)) —
**o modelo que os cinco módulos de método explicam**. Este caderno treina,
mede, escolhe o paciente-exemplo por regra e arma as armadilhas que os
módulos vão usar.

Roda inteiro da **amostra commitada** (val e teste inteiros; treino 200 mil
de 1,24 M): sem rede, sem binário de modelo no repositório — cada módulo
refaz o fit em segundos e explica o MESMO modelo por construção. O delta
amostra→base cheia está no
[internals](srag_model_internals.ipynb).

In [1]:
%pip install -q pandas pyarrow scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "README.md").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "README.md").exists():
    import tempfile
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import srag_60_model as M

g = M.load_gold(REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet")
print(f"amostra: {len(g):,} linhas × {len(g.columns)} colunas")
print(dict(g["split"].value_counts()))

amostra: 240,290 linhas × 50 colunas
{'train': np.int64(200000), 'val': np.int64(24148), 'test': np.int64(16142)}


## 1. O Ouro em uma tela

O funil e as decisões vivem no manifesto; este caderno só as lê do
`counts.json` commitado — custo de CI zero, mesmo padrão do internals do
Prata com o PROFILE.json.

In [3]:
counts = json.loads(
    (REPO / "modules/00-dataset/gold/counts.json").read_text(encoding="utf-8")
)
print("funil:")
for nome, n in counts["funil"]:
    print(f"  {nome:<38} {n:>11,}")
print()
for s, v in counts["splits"].items():
    print(f"  {s:<6} {v['n']:>11,}   óbitos {v['obitos']:>8,}   CFR {v['letalidade_pct']}%")

funil:
  prata (sem linhas deslocadas)            4,109,560
  coorte_hospitalizado                     2,428,695
  covid_caso (definição ampla)             1,364,487
  casos fechados (EVOLUCAO 1 ou 2)         1,283,801
  DT_SIN_PRI >= 2020-02-26                 1,283,745
  idade presente                           1,282,973
  idade <= 120                             1,282,970

  test        16,142   óbitos    2,945   CFR 18.2%
  train    1,242,680   óbitos  390,769   CFR 31.4%
  val         24,148   óbitos    4,572   CFR 18.9%


A letalidade cai de 31,4% (treino) para 18,2% (teste): **o split
temporal expõe a deriva de regime de propósito** — treinamos no passado e
testamos num mundo que mudou. É o achado central, não um incômodo.

## 2. Os dois modelos

In [4]:
t0 = time.time()
xgb, logit = M.fit_models(g)
print(f"fit dos dois modelos: {time.time() - t0:.1f}s")
linhas = []
for nome, modelo in (("XGBoost", xgb), ("logística", logit)):
    for s in ("val", "test"):
        linhas.append({"modelo": nome} | M.metrics(modelo, g, s))
print(pd.DataFrame(linhas).to_string(index=False))

fit dos dois modelos: 4.2s
   modelo split     n  obitos    auc  brier  previsto_medio  observado
  XGBoost   val 24148    4572 0.7564 0.1356          0.2209     0.1893
  XGBoost  test 16142    2945 0.7644 0.1308          0.2154     0.1824
logística   val 24148    4572 0.7206 0.1550          0.2764     0.1893
logística  test 16142    2945 0.7246 0.1452          0.2419     0.1824


O XGBoost vence a logística por ~3 pontos de AUC — e **ambos superestimam
o risco** no teste (previsto ~0,22 contra observado 0,18): aprenderam as
letalidades de 2020–22. A calibração é a história, não o defeito:

In [5]:
m = (g["split"] == "test").to_numpy()
p = M.predict_proba(xgb, g[m])
y = g.loc[m, "y_obito"].to_numpy()
faixas = pd.cut(p, np.linspace(0, 1, 11))
conf = pd.DataFrame({"faixa": faixas, "previsto": p, "observado": y}).groupby(
    "faixa", observed=True
).agg(n=("observado", "size"), previsto=("previsto", "mean"),
      observado=("observado", "mean")).round(3)
print(conf.to_string())
print(f"\ngap médio (previsto − observado): {p.mean() - y.mean():+.3f}")

               n  previsto  observado
faixa                                
(0.0, 0.1]  4543     0.043      0.025
(0.1, 0.2]  3360     0.152      0.118
(0.2, 0.3]  3657     0.248      0.207
(0.3, 0.4]  2543     0.346      0.310
(0.4, 0.5]  1415     0.444      0.400
(0.5, 0.6]   488     0.539      0.514
(0.6, 0.7]   117     0.638      0.538
(0.7, 0.8]    18     0.733      0.556
(0.8, 0.9]     1     0.868      1.000

gap médio (previsto − observado): +0.033


## 3. O paciente-exemplo — uma regra, não um índice

A era Breast Cancer tinha o "paciente #67"; aqui a escolha é **regra**:
|p − 0,5| mínimo no teste, empate por `gold_id`. Determinística, aplicada
ao modelo-da-amostra — o mesmo que todos os módulos usam.

In [6]:
ex = M.pick_exemplar(xgb, g)
p21 = M.paciente_2021(xgb, g)
mostrar = ["idade_anos", "cs_sexo", "cs_raca", "regiao", "capital_interior",
           "ano_onset", "semana_epi", "n_doses_antes_do_sintoma",
           "vacina_covid_declarada", "diabetes", "cardiopati", "obesidade",
           "saturacao", "dispneia", "nosocomial", "p_obito"]
quadro = pd.DataFrame({"exemplar (teste)": ex[mostrar], "paciente 2021": p21[mostrar]})
print(quadro.to_string())
print(f"\ngold_id: exemplar={int(ex.gold_id)}   2021={int(p21.gold_id)}")

                         exemplar (teste) paciente 2021
idade_anos                      89.694733     44.104038
cs_sexo                         masculino      feminino
cs_raca                             parda        branca
regiao                                 SE            SE
capital_interior                 interior       capital
ano_onset                            2024          2021
semana_epi                             34            18
n_doses_antes_do_sintoma                1             0
vacina_covid_declarada               True         False
diabetes                              nao           nao
cardiopati                            sim           nao
obesidade                             nao           sim
saturacao                             sim           sim
dispneia                              sim           sim
nosocomial                            nao      ignorado
p_obito                          0.500003      0.500001

gold_id: exemplar=1276776   2021=834192


Dois pacientes, mesma regra, dois regimes — o segundo existe para os
módulos mostrarem o mesmo método sob outra época.

## 4. As quatro impossibilidades, armadas

`gate_impossible()` centraliza o que os cinco módulos vão contar: uma
perturbação que cruza uma destas fronteiras fabrica um paciente que **não
pode existir** — e a conta é a mesma função em todos os módulos, então os
números são idênticos por construção.

In [7]:
imp = {
    "portão do funil (sem fator de risco declarado)":
        (~g["fator_risc_portao"].astype(bool)),
    "pré-campanha (nenhuma dose podia existir)":
        (g["meses_desde_mar2020"] < M.MESES_CAMPANHA),
    "critério-2 por um fio (só tosse OU só garganta)": (g["n_crit2"] == 1),
    "critério-3 por um fio": (g["n_crit3"] == 1),
}
n = len(g)
for nome, mask in imp.items():
    print(f"  {nome:<48} {int(mask.sum()):>7,}  ({100 * int(mask.sum()) / n:5.2f}%)")
reais = M.gate_impossible(g)
print(f"\nlinhas REAIS que a função marca: {int(reais.sum())}")
if reais.any():
    r = g[reais].iloc[0]
    print(f"  → gold_id {int(r.gold_id)}: início {r.ano_onset} (pré-campanha) com "
          f"{int(r.n_doses_antes_do_sintoma)} dose registrada — uma das datas de dose "
          f"impossíveis que o módulo 00 flagra e não conserta (internals do Prata §6).")

  portão do funil (sem fator de risco declarado)    89,550  (37.27%)
  pré-campanha (nenhuma dose podia existir)         71,283  (29.67%)
  critério-2 por um fio (só tosse OU só garganta)  191,281  (79.60%)
  critério-3 por um fio                             59,288  (24.67%)



linhas REAIS que a função marca: 1
  → gold_id 190127: início 2020 (pré-campanha) com 1 dose registrada — uma das datas de dose impossíveis que o módulo 00 flagra e não conserta (internals do Prata §6).


## 5. A célula-armadilha: por que a imagem ficou fora

A pergunta do curso: *"por que não deixar a tomografia como feature e o
peso zerar?"* Porque modelo não zera proxy — usa. E o que a medição mostra
é ainda melhor que o esperado:

In [8]:
tomo = g["tomo_res"].astype(str)
tem = tomo != "desconhecido"
y_all = g["y_obito"]
print(f"tomografia registrada: {100 * int(tem.sum()) / len(g):.2f}% da amostra")
print(f"CFR com registro: {100 * y_all[tem].mean():.1f}%   sem registro: "
      f"{100 * y_all[~tem].mean():.1f}%")
por_nivel = y_all.groupby(tomo).agg(["size", "mean"])
por_nivel.columns = ["n", "CFR"]
print((por_nivel.assign(CFR=lambda d: (100 * d.CFR).round(1))
       .sort_values("CFR", ascending=False)).to_string())

tomografia registrada: 63.02% da amostra
CFR com registro: 28.5%   sem registro: 30.5%
                  n   CFR
tomo_res                 
9             11342  33.4
desconhecido  88867  30.5
1             80677  28.6
6             44189  28.1
2              4816  27.7
3              2541  25.5
5              7289  25.1
4               569  12.1


In [9]:
com_img = g.copy()
extra = ["raiox_res", "tomo_res"]
for c in extra:
    com_img[c] = com_img[c].astype("category")

import xgboost as xgb_lib
x_xgb, _, y_vec = M.design_matrix(g)
x_img = pd.concat([x_xgb, com_img[extra]], axis=1)
tr = (g["split"] == "train").to_numpy()
te = (g["split"] == "test").to_numpy()
m_img = xgb_lib.XGBClassifier(**M.XGB_PARAMS)
m_img.fit(x_img[tr], y_vec[tr])
from sklearn.metrics import roc_auc_score
auc_sem = M.metrics(xgb, g, "test")["auc"]
auc_com = round(float(roc_auc_score(y_vec[te], m_img.predict_proba(x_img[te])[:, 1])), 4)
ganho = pd.Series(m_img.get_booster().get_score(importance_type="gain"))
print(f"AUC teste sem imagem: {auc_sem}   com imagem: {auc_com}   Δ = {auc_com - auc_sem:+.4f}")
print("\nposto de ganho das colunas de imagem entre as features:")
ranking = ganho.sort_values(ascending=False).reset_index()
ranking.columns = ["feature", "ganho"]
print(ranking[ranking["feature"].isin(extra)].to_string())
print(f"(de {len(ranking)} features usadas)")

AUC teste sem imagem: 0.7644   com imagem: 0.7682   Δ = +0.0038

posto de ganho das colunas de imagem entre as features:
      feature      ganho
27  raiox_res  15.918290
28   tomo_res  12.945128
(de 41 features usadas)


A armadilha medida é **invertida**: a *ausência* do registro acompanha
mais óbito do que qualquer resultado do exame, e `9-Ignorado` é o nível
mais letal — qualidade de documentação vazando no rótulo, não pulmão. Um
SHAP sobre isso produziria a explicação clinicamente enganosa "fazer
tomografia protege". Por isso a imagem viaja no Ouro **como diagnóstico**,
nunca como feature — e esta célula existe para a aula.

## 6. O que este caderno entrega aos módulos 01–05

- `M.load_gold()` — a mesma amostra para todos;
- `M.fit_models()` — o mesmo XGBoost (e a logística de contraste);
- `M.pick_exemplar()` / `M.paciente_2021()` — os mesmos dois pacientes;
- `M.gate_impossible()` — as mesmas quatro contagens;
- [`MODEL.md`](../MODEL.md) — o model card gerado, com as métricas desta
  rodada (verificado por `--check-card`).

O que fica declarado como limite: o fit cheio (1,24 M) e o delta para a
amostra vivem no [internals](srag_model_internals.ipynb); imagem fora; sem
pesos de classe (manifesto §2.7).